# Chapitre 1 — Lire les données · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch01_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre, de haut en bas : tant qu'une réponse manque, les cellules de vérification le signalent sans erreur. Le corrigé se trouve dans `ch01_corrige.ipynb`, sur la même page compagnon.

**Convention.** Comme dans le livre, la variance, l'écart-type, le score z et la covariance divisent par $n$ (`ddof=0`), sauf mention explicite.

**Sommaire.** Données · Les calculs des leçons · Exercices 1.1 à 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Couleurs du livre : une couleur = un rôle (observé, prédit, écart, droite du collègue)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Données

Les identifiants (D0.1, D1.1…) sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 1, recopiées des fichiers CSV du livre (ne pas modifier)

# D0.1 — les six logements de l'ouverture
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — prix de vente des T2 (k€), leçon 1.1
nord = [174, 177, 181, 183, 185]
sud  = [150, 165, 185, 195, 205]

# D1.2 — registre des ventes du quartier Sud, leçons 1.2 à 1.5
registre = pd.DataFrame({
    'vente':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'pieces':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'surface_m2':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'dpe':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'prix_kEUR':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'delai_jours': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — prix au m² selon l'étage (k€/m²), leçon 1.5
etages = pd.DataFrame({
    'etage':        [0, 1, 2, 3, 4, 5, 6],
    'prix_m2_kEUR': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — délais de vente de trois agences (jours), exercice 1.1
delais_agences = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — loyers mensuels des studios (€), exercice 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

In [ ]:
# Outils fournis : vous n'avez pas à les modifier

def quartiles_livre(valeurs):
    """Q1 et Q3 selon la règle du livre (équation 1.5) : rangs ⌈n/4⌉ et ⌈3n/4⌉."""
    x = sorted(valeurs)
    n = len(x)
    rang_q1 = -(-n // 4)          # arrondi à l'entier supérieur de n/4
    rang_q3 = -(-3 * n // 4)      # arrondi à l'entier supérieur de 3n/4
    return x[rang_q1 - 1], x[rang_q3 - 1]   # rang 1 = position 0 en Python


def points_par_rangee(series, xlabel, bandes=None, ax=None):
    """Une rangée de points par série, comme les figures de la leçon 1.1.
    bandes : dictionnaire {nom: (moyenne, écart-type)} pour dessiner moyenne ± écart-type."""
    if ax is None:
        fig, ax = plt.subplots()
    noms = list(series)
    couleurs = [BLEU, ORANGE, '#52514e']
    for i, nom in enumerate(noms):
        y = len(noms) - 1 - i
        if bandes and nom in bandes:
            m, s = bandes[nom]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[nom], [y] * len(series[nom]), s=50,
                   color=couleurs[i % 3], zorder=3)
    ax.set_yticks(range(len(noms)), list(reversed(noms)))
    ax.set_ylim(-0.6, len(noms) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 1.1 à 1.5. Exécutez-les et comparez avec le livre. Ici, `np.std` et `np.var` sont appelés avec `ddof=0`, c'est-à-dire avec une division par $n$, comme dans le chapitre.

In [ ]:
# Leçon 1.1 — moyenne, variance et écart-type (division par n)
for nom, serie in [('Nord', nord), ('Sud', sud)]:
    print(f'{nom} : moyenne = {np.mean(serie):g} k€, variance = {np.var(serie, ddof=0):g} k€², '
          f'écart-type = {np.std(serie, ddof=0):g} k€')

In [ ]:
# Leçon 1.2 — effectifs par classe DPE (variable qualitative ordinale)
registre['dpe'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Leçon 1.3 — moyenne, médiane et quartiles des délais
d = registre['delai_jours']
print('moyenne :', d.mean(), ' médiane :', d.median())
print('sans S13 : moyenne', d[registre['vente'] != 'S13'].mean(),
      ' médiane', d[registre['vente'] != 'S13'].median())
print('quartiles, règle du livre (équation 1.5) :', quartiles_livre(d))
print('quartiles, NumPy par défaut (interpolation) :', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Leçon 1.4 — scores standardisés (équation 1.6, écart-type avec division par n)
for nom, serie, vente in [('Nord', nord, 188), ('Sud', sud, 210)]:
    m, s = np.mean(serie), np.std(serie, ddof=0)
    print(f'{nom} : z de la série = {np.round((np.array(serie) - m) / s, 4)}, '
          f'z de la vente à {vente} k€ = {(vente - m) / s:g}')

In [ ]:
# Leçon 1.5 — covariance et corrélation des cinq T2 du Sud (division par n)
t2 = registre[registre['pieces'] == 2]
x, y = t2['surface_m2'].to_numpy(float), t2['prix_kEUR'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covariance :', cov, 'm²·k€   r :', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r par np.corrcoef :', np.corrcoef(x, y)[0, 1].round(4))
print('r étage / prix au m² :', np.corrcoef(etages['etage'], etages['prix_m2_kEUR'])[0, 1].round(4))

## Exercice 1.1 — Lire un graphique

*Données D1.E1. Notions : moyenne et écart-type (leçon 1.1).* Le graphique ci-dessous reproduit celui de l'exercice 1.1 du livre.

In [ ]:
points_par_rangee({f'Agence {a}': v for a, v in delais_agences.items()},
                  'Délai de vente (jours)')
plt.xlim(0, 60)
plt.show()

**a) et b)** Estimez à l'œil le délai moyen de chaque agence, puis rangez les agences de la moins dispersée à la plus dispersée.

In [ ]:
# À COMPLÉTER : vos estimations à l'œil, avant tout calcul
moyenne_estimee = {'P': None, 'Q': None, 'R': None}   # en jours
ordre_dispersion = None   # liste, de la moins à la plus dispersée, ex. ['Q', 'P', 'R']

In [ ]:
# Vérification (division par n)
for a, v in delais_agences.items():
    print(f'Agence {a} : moyenne = {np.mean(v):g} j, écart-type = {np.std(v, ddof=0):.2f} j')
ordre_reel = sorted(delais_agences, key=lambda a: np.std(delais_agences[a]))
if ordre_dispersion is None or None in moyenne_estimee.values():
    print('Complétez la cellule précédente.')
else:
    print('Ordre de dispersion :', 'correct' if ordre_dispersion == ordre_reel else 'à revoir')

**c)** Comptez, pour Q et pour R, les ventes conclues à 2 jours ou moins de la moyenne.

In [ ]:
# À COMPLÉTER : vos comptages lus sur le graphique
proches_Q = None
proches_R = None

In [ ]:
# Vérification
for a, la_votre in [('Q', proches_Q), ('R', proches_R)]:
    v = np.array(delais_agences[a])
    vrai = int(np.sum(np.abs(v - v.mean()) <= 2))
    if la_votre is None:
        print(f'Agence {a} : pas encore complété.')
    else:
        print(f"Agence {a} : {'correct' if la_votre == vrai else 'à revoir'}")

*Votre explication pour c) :* …

## Exercice 1.2 — Deux résidences, un même loyer moyen

*Données D1.E2. Notions : moyenne, variance et écart-type (leçon 1.1).* Camille ne peut pas dépasser 570 € par mois. Les questions a) à d) se font d'abord sur papier, avec le tableau de l'exercice 1.2 du livre. Ce notebook sert à vérifier vos réponses, puis à programmer l'écart-type (question e) et à comparer deux conventions (question f).

In [ ]:
points_par_rangee({'Les Tilleuls': tilleuls, 'Le Port': port}, "Loyer mensuel d'un studio (€)")
plt.axvline(570, color='black', linestyle=':', label='budget de Camille : 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Lire les loyers

Combien de studios Camille peut-elle louer dans chaque résidence avec 570 €, puis avec 620 € ?

In [ ]:
# À COMPLÉTER : nombre de studios accessibles, lu dans la liste des loyers
accessibles = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Vérification : `<=` produit des vrai/faux, np.sum compte les « vrai »
for budget, reponses in accessibles.items():
    for nom, serie in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        vrai = int(np.sum(np.array(serie) <= budget))
        la_votre = reponses[nom]
        etat = 'pas encore complété' if la_votre is None else ('correct' if la_votre == vrai else 'à revoir')
        print(f'Budget {budget} €, {nom} : {etat}')

*Votre réponse à a), en une phrase :* …

### b) Les deux moyennes

Calculez-les sur papier, puis reportez vos résultats ci-dessous.

In [ ]:
# À COMPLÉTER : vos moyennes calculées à la main (en €)
moyenne_tilleuls_main = None
moyenne_port_main = None

In [ ]:
# Vérification
for nom, serie, la_votre in [('Les Tilleuls', tilleuls, moyenne_tilleuls_main),
                              ('Le Port', port, moyenne_port_main)]:
    if la_votre is None:
        print(f'{nom} : complétez la cellule précédente.')
    else:
        ok = np.isclose(la_votre, np.mean(serie))
        print(f"{nom} : {'correct' if ok else 'à revoir'} (votre valeur : {la_votre} €)")

### c) Variance et écart-type à la main (division par $n = 6$)

Sur papier, dressez pour chaque résidence le tableau des écarts à la moyenne et de leurs carrés (modèle : le tableau du calcul pas à pas de la leçon 1.1). Reportez ici les quatre résultats.

In [ ]:
# À COMPLÉTER : vos résultats calculés à la main (division par n)
variance_tilleuls_main = None   # en €²
variance_port_main = None       # en €²
ecart_type_tilleuls_main = None # en €
ecart_type_port_main = None     # en €

In [ ]:
# Vérification (division par n : ddof=0)
vos_valeurs = {
    'variance Tilleuls': (variance_tilleuls_main, np.var(tilleuls, ddof=0)),
    'variance Port': (variance_port_main, np.var(port, ddof=0)),
    'écart-type Tilleuls': (ecart_type_tilleuls_main, np.std(tilleuls, ddof=0)),
    'écart-type Port': (ecart_type_port_main, np.std(port, ddof=0)),
}
for nom, (la_votre, attendu) in vos_valeurs.items():
    if la_votre is None:
        print(f'{nom} : pas encore complété.')
    else:
        print(f"{nom} : {'correct' if np.isclose(la_votre, attendu) else 'à revoir'}")

### d) Interpréter

*Votre réponse :* …

### e) Programmer l'écart-type

Complétez la fonction en suivant l'équation 1.3 du livre, avec une division par $n$ :

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

N'utilisez ni `np.std` ni `np.var` : le but est d'écrire chaque étape vous-même.

In [ ]:
# Exercice 1.2 e) — À COMPLÉTER
def ecart_type(valeurs):
    """Écart-type, division par n (équation 1.3)."""
    x = np.array(valeurs, dtype=float)
    n = len(x)
    moyenne = None   # À COMPLÉTER : équation 1.1
    ecarts = None    # À COMPLÉTER : un écart par valeur
    variance = None  # À COMPLÉTER : moyenne des carrés
    return None      # À COMPLÉTER : retour à l'unité

<details><summary><b>Indice</b> (cliquez si vous bloquez)</summary>

Avec NumPy, une opération écrite sur un tableau s'applique à chaque valeur : `x - moyenne` donne d'un coup les six écarts, et `ecarts ** 2` leurs six carrés. `np.sum(...)` additionne, `np.sqrt(...)` prend la racine carrée.

</details>

In [ ]:
# Vérification : exécutez cette cellule sans la modifier
if ecart_type(tilleuls) is None:
    print('La fonction renvoie encore None : complétez-la.')
else:
    for nom, serie, attendu in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtenu = ecart_type(serie)
        etat = 'correct' if np.isclose(obtenu, attendu) else f'à revoir (attendu : {attendu})'
        print(f'{nom} : {obtenu:.2f} € → {etat}')

### f) Approfondissement : diviser par $n$ ou par $n - 1$

Exécutez la cellule ci-dessous. Retrouvez 10,95 à la main en divisant la somme des carrés par $n - 1 = 5$. Quelle option de pandas donne la convention du chapitre ?

*Votre réponse :* …

In [ ]:
loyers = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, par défaut (division par n − 1) :', loyers.std().round(2).to_dict())
print('pandas, ddof=0 (division par n)          :', loyers.std(ddof=0).round(2).to_dict())

## Exercice 1.3 — Une vente surprenante ?

*Données D1.1. Notions : score standardisé (leçon 1.4, équation 1.6).* Un T2 vient de se vendre 196 k€. Moyennes et écarts-types : ceux de la leçon 1.1, avec division par $n$.

In [ ]:
# À COMPLÉTER : vos calculs faits sur papier
z_nord = None
z_sud = None
prix_sud_aussi_exceptionnel = None   # en k€, question c)

In [ ]:
# Vérification (écart-type avec division par n)
attendus = {
    'z Nord': (z_nord, (196 - np.mean(nord)) / np.std(nord, ddof=0)),
    'z Sud': (z_sud, (196 - np.mean(sud)) / np.std(sud, ddof=0)),
    'prix Sud équivalent': (prix_sud_aussi_exceptionnel,
                            np.mean(sud) + (196 - np.mean(nord)) / np.std(nord, ddof=0) * np.std(sud, ddof=0)),
}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'pas encore complété' if la_votre is None else ('correct' if np.isclose(la_votre, vrai) else 'à revoir')
    print(f'{nom} : {etat}')

*Votre réponse à b) :* …

## Exercice 1.4 — Surface et prix vont-ils ensemble ?

*Données D0.1 (les six logements de l'ouverture). Notions : covariance et corrélation (leçon 1.5, équations 1.7 et 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(logements['surface_m2'], logements['prix_kEUR'], color=BLEU)
for _, l in logements.iterrows():
    plt.annotate(l['logement'], (l['surface_m2'], l['prix_kEUR']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Surface (m²)')
plt.ylabel('Prix de vente (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Avant tout calcul : $r$ sera-t-il positif ou négatif ? Proche de 0, de 0,5 ou de 1 ?

*Votre réponse :* …

**b)** Complétez les lignes ci-dessous. Utilisez une division par $n$ partout.

In [ ]:
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
n = len(x)

produits = None     # À COMPLÉTER : (x_i − x̄)(y_i − ȳ) pour chaque logement
covariance = None   # À COMPLÉTER : équation 1.7, division par n
r = None            # À COMPLÉTER : équation 1.8 (écarts-types avec ddof=0)

<details><summary><b>Indice</b></summary>

`x.mean()` donne la moyenne, `x - x.mean()` les six écarts. Le produit `*` de deux tableaux multiplie terme à terme. `x.std(ddof=0)` donne l'écart-type avec division par $n$.

</details>

In [ ]:
# Vérification
if r is None or covariance is None:
    print('Complétez la cellule précédente.')
else:
    print('covariance :', round(covariance, 2), 'm²·k€ →',
          'correct' if np.isclose(covariance, 1637.5) else 'à revoir (attendu : 1637,5)')
    print('r :', round(r, 4), '→', 'correct' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'à revoir')

**c)** Corrélation proche de 1, mais écarts de 5 à 11 k€ en valeur absolue : est-ce contradictoire ?

*Votre réponse :* …

In [ ]:
print('np.cov par défaut (division par n − 1) :', np.cov(x, y)[0, 1])
print('np.cov avec bias=True (division par n)  :', np.cov(x, y, bias=True)[0, 1])

## Exercice 1.5 — Prédire, puis vérifier

*Données D1.1. Notions : moyenne et écart-type (leçon 1.1), médiane et écart interquartile (leçon 1.3).* Un sixième T2 se vend au Nord pour 240 k€.

**a)** Avant tout calcul, notez vos prédictions : la moyenne, la médiane, l'écart-type et l'écart interquartile augmentent-ils beaucoup, un peu ou pas du tout ?

*Vos prédictions :* …

In [ ]:
nord_plus = nord + [240]
print('Nord :', sorted(nord))
print('Nord avec la vente à 240 k€ :', sorted(nord_plus))

**b)** Complétez la fonction `resume`. Elle doit renvoyer les quatre indicateurs, avec l'écart-type divisé par $n$ et les quartiles calculés par la règle du livre (`quartiles_livre`, fournie plus haut).

In [ ]:
# À COMPLÉTER
def resume(serie):
    q1, q3 = quartiles_livre(serie)
    return {
        'moyenne': None,              # À COMPLÉTER
        'médiane': None,              # À COMPLÉTER : np.median
        'écart-type': None,           # À COMPLÉTER : division par n
        'écart interquartile': None,  # À COMPLÉTER : à partir de q1 et q3
    }

In [ ]:
tableau = pd.DataFrame({'Nord (5 ventes)': resume(nord), 'Nord + 240 k€ (6 ventes)': resume(nord_plus),
                        'Sud (5 ventes)': resume(sud)})
if tableau.isna().any().any():
    print('Complétez la fonction resume.')
else:
    attendu = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (avant, apres) in attendu.items():
        ok = np.isclose(tableau.loc[k].iloc[0], avant) and np.isclose(tableau.loc[k].iloc[1], apres, atol=0.005)
        print(f"{k} : {'correct' if ok else 'à revoir'}")
tableau.round(2)

In [ ]:
# Les deux séries sur le modèle de la figure « moyenne ± écart-type » de la leçon 1.1
if tableau.isna().any().any():
    print('Complétez la fonction resume pour afficher le graphique.')
else:
    points_par_rangee({'Nord': nord, 'Nord + 240': nord_plus},
                      'Prix de vente d\'un T2 (k€)',
                      bandes={'Nord': (np.mean(nord), np.std(nord, ddof=0)),
                              'Nord + 240': (np.mean(nord_plus), np.std(nord_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** Les prix du Nord sont-ils devenus plus imprévisibles que ceux du Sud ?

*Votre réponse :* …

## Exercice 1.6 — Défi : deux droites, une même erreur moyenne

*Données D0.1. Notions : écart signé et valeur absolue (ouverture, leçon 1.1), écart-type (leçon 1.1).* Droite tracée à l'œil : prix prédit = 3 × surface + 30. Droite paresseuse : prix prédit = 222 k€ pour tous les logements.

In [ ]:
prix = logements['prix_kEUR'].to_numpy(dtype=float)
surface = logements['surface_m2'].to_numpy(dtype=float)

ecarts_oeil = None          # À COMPLÉTER : observé − prédit, droite tracée à l'œil
ecarts_paresseuse = None    # À COMPLÉTER : observé − prédit, droite paresseuse

In [ ]:
# Les trois indicateurs de la question b) (écart-type : division par n)
if ecarts_oeil is None or ecarts_paresseuse is None:
    print('Complétez la cellule précédente.')
else:
    indicateurs = pd.DataFrame({
        nom: {'moyenne des écarts signés': e.mean(),
              'moyenne des écarts en valeur absolue': np.abs(e).mean(),
              'écart-type des écarts signés': e.std(ddof=0)}
        for nom, e in [('droite à l\'œil', ecarts_oeil), ('droite paresseuse', ecarts_paresseuse)]
    })
    print('écarts, droite à l\'œil   :', ecarts_oeil)
    print('écarts, droite paresseuse :', ecarts_paresseuse)
    print('vérification :', 'correct' if np.allclose(indicateurs.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicateurs.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'à revoir')
    display(indicateurs.round(2))

In [ ]:
if ecarts_oeil is not None and ecarts_paresseuse is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(surface - 0.8, ecarts_oeil, color=ORANGE, label="droite à l'œil")
    ax.scatter(surface + 0.8, ecarts_paresseuse, color=BLEU, marker='s', label='droite paresseuse')
    ax.set_xlabel('Surface (m²)')
    ax.set_ylabel('Écart observé − prédit (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Vos réponses à c) et d) :* …

In [ ]:
# Contrôle de d) : l'écart-type des écarts de la droite paresseuse égale celui des prix
if ecarts_paresseuse is not None:
    print(round(ecarts_paresseuse.std(ddof=0), 4), round(prix.std(ddof=0), 4))